# Tarea 2 · Parte 2 — Evaluación y extracción estructurada

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

Este notebook mide qué tan bien leen los modelos (CER en documentos, WER en audio), los compara sobre **las mismas muestras**, extrae campos estructurados con un LLM y audita esa extracción a mano.

| | |
|---|---|
| **Fecha** | 27 de octubre de 2026 |
| **Datos** | `datos/documentos.csv` (25 documentos) y `datos/transcripciones.csv` (20 clips, 32.3 min) de la Parte 1 |
| **Referencias** | Transcripción manual propia de 10 documentos y 10 clips (`datos/raw/referencias/`, no versionada: puede tener nombres reales) |

## 1. Muestra de evaluación y configuraciones (fijadas antes de ver métricas)

Esta sección se escribió y se subió a GitHub **antes** de terminar las transcripciones de referencia y antes de calcular cualquier CER o WER. Ninguna configuración se cambia después según los resultados.

### Muestra

- **Documentos (10):** 3 impresos, 3 fotos y 4 manuscritos. Los manuscritos tienen un elemento más porque son el caso difícil.
- **Audios (10):** 7 fuentes o autores distintos y 5 condiciones: calle, fiesta con ruido, marcha con ruido, testimonio y lectura en estudio. Incluye clips con 2 o más hablantes.
- **Criterio:** dentro de cada tipo se eligieron los elementos de **menor costo de transcripción manual**. Así quedaron fuera `doc_003`, una página casi vacía, y las fotos de más de 5 000 caracteres (`doc_021`–`doc_025`). En audio se excluyeron la conferencia (`clip_003`) y los testimonios de 3–4 min (`clip_014`, `clip_015`), y por la misma razón se tomaron 2 de las 6 lecturas de OpenSLR.
- **Sesgo conocido:** el criterio favorece páginas y audios cortos. Las métricas describen esta muestra, no necesariamente a las páginas de periódico más densas.

### Transcripción de referencia
Se hace a mano y sin mirar la salida de ningún modelo. Se copia línea por línea y con la ortografía original. Las palabras ilegibles se marcan `[?]` y los tramos que no se entienden, `[inaudible]`. En audio se omiten las muletillas (`eh`, `mmm`) y los números se escriben con cifras. La guía completa está en `datos/raw/referencias/LEEME.md`.

### Normalización (la misma para referencia e hipótesis)
`voz.normalizar`: minúsculas, Unicode NFC, sin puntuación y con espacios simples. **Las tildes se conservan.** Antes se eliminan las marcas `[HABLANTE_n]`, `[?]` y `[inaudible]`. CER y WER se calculan **sobre los textos originales**, sin seudonimizar, para que el reemplazo de nombres no cuente como error. El valor global agrega los errores de los 10 casos (jiwer) y también se reporta el valor de cada caso.

### Modelos

| Modalidad | Rol | Configuración |
|---|---|---|
| OCR | **Línea base** | PaddleOCR 3.0.3 por defecto: PP-OCRv5 *mobile* det + rec, orientación de página, desdoblado UVDoc, orientación de línea y orden de lectura de Paddle. Solo se cambian los modelos *server* por los *mobile*, porque los *server* no corren en este CPU (Parte 1). |
| OCR | Ajustado | PaddleOCR de la Parte 1: *mobile*, detección a 1600 px, sin preprocesado de página y orden de lectura por columnas. Se definió en la Parte 1, antes de que existieran las referencias. |
| OCR | Alternativo | Tesseract 5.5.1 (tesserocr 2.9.1, `spa` de tessdata 4.1.0, LSTM, PSM 3) |
| ASR | **Línea base** | faster-whisper `small` (int8, CPU, `language=es`, VAD Silero, beam 5) |
| ASR | Mejorado | faster-whisper `large-v3-turbo`, con la misma configuración (el modelo de la Parte 1) |
| ASR | Alternativo | Qwen3-ASR 0.6B (transformers, float32, CPU, `language=es`, tramos VAD ≤ 12 s, decodificación greedy) |

La latencia de ASR se mide solo durante la transcripción, sin contar la carga del modelo ni la diarización. Los modelos se ejecutan uno por uno en el mismo equipo (Intel i5 de 2 núcleos). **Factor de tiempo real = segundos de proceso / duración del audio.**

### Extracción con LLM
- **Modelo:** Gemini 3.8 Flash (`gemini-3.8-flash`, API gratuita), temperatura 0 y salida JSON con esquema fijo.
- **Entrada:** los textos **ya seudonimizados**, para no enviar nombres reales a un servicio externo.
- **Campos comunes:** `modalidad`, `tema` y `resumen`.
- **Campos de documentos:** `emisor`, `fecha`, `lugar` y `monto`.
- **Campos de audio:** `numero_hablantes` y `citas_clave`.
- Un campo que no aparece en el texto debe quedar en `null`.

La auditoría humana revisa 20 extracciones sorteadas con semilla fija (10 documentos y 10 audios).

In [1]:
import pandas as pd

docs = pd.read_csv("datos/documentos.csv")
clips = pd.read_csv("datos/transcripciones.csv")
fuentes = pd.read_csv("datos/fuentes.csv").set_index("id")

MUESTRA_DOCS = ["doc_001", "doc_002", "doc_004", "doc_018", "doc_019", "doc_020", "doc_009", "doc_015", "doc_016", "doc_017"]
MUESTRA_CLIPS = ["clip_001", "clip_010", "clip_011", "clip_005", "clip_006", "clip_007", "clip_008", "clip_013", "clip_016", "clip_017"]

m_docs = docs.set_index("id").loc[MUESTRA_DOCS, ["tipo"]].assign(caracteres_ocr=docs.set_index("id")["texto_ocr"].str.len())
m_clips = clips.set_index("id").loc[MUESTRA_CLIPS, ["duracion_seg", "n_hablantes"]].join(fuentes[["tipo", "autor"]])
print("Documentos por tipo:", m_docs["tipo"].value_counts().to_dict())
print(f"Audios: {len(m_clips)} clips, {m_clips['duracion_seg'].sum() / 60:.1f} min, {m_clips['autor'].nunique()} fuentes, condiciones:", sorted(m_clips["tipo"].unique()))
display(m_docs, m_clips)

Documentos por tipo: {'manuscrito': 4, 'impreso': 3, 'foto': 3}
Audios: 10 clips, 11.5 min, 7 fuentes, condiciones: ['calle', 'fiesta, ruido', 'lectura, estudio', 'marcha, ruido', 'testimonio']


,tipo,caracteres_ocr
id,,
doc_001,impreso,362
doc_002,impreso,1038
doc_004,impreso,1516
doc_018,foto,2526
doc_019,foto,2388
doc_020,foto,504
doc_009,manuscrito,1892
doc_015,manuscrito,1861
doc_016,manuscrito,1144


,duracion_seg,n_hablantes,tipo,autor
id,,,,
clip_001,136.3,2,calle,JorgeFlores-pe
clip_010,96.1,1,calle,Gianpier Yair Venegas Ambia
clip_011,77.4,2,calle,Gianpier Yair Venegas Ambia
clip_005,81.6,2,"fiesta, ruido",Rashelley Perez
clip_006,77.4,2,"marcha, ruido",Hellloff233
clip_007,95.0,5,"marcha, ruido",Hellloff233
clip_008,54.3,3,"marcha, ruido",Joaquin Canales
clip_013,53.1,1,testimonio,Chantell Acosta
clip_016,10.2,1,"lectura, estudio","Google (Guevara-Rukoz et al., 2020)"
